
**Input utama**
- `data/labels/batch2_labeled_combined.csv`
- `data/labels/batch3_labeled_combined.csv`
- `data/features/batch2_account_features_with_scores.csv`
- `data/features/batch3_account_features_with_scores.csv`

**Output utama**
- `output/phase5/` untuk tabel, confusion matrix, dan grafik
- `reports/BASELINE_EVALUATION_REPORT.md`
- `models/phase5_baseline_rf.pkl`

Catatan: `buzzer_score` tetap dipertahankan sebagai metadata unsupervised, tetapi tidak dipakai sebagai ground truth dan tidak dimasukkan ke fitur model baseline.

## 1. Setup dan reproducibility

Path dicari relatif terhadap repository agar notebook dapat dijalankan dari root project maupun dari folder `notebooks/`. Semua seed dan direktori output ditetapkan di awal.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import pickle
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")

SEED = 42
np.random.seed(SEED)

def find_repo_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "PRD.md").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Repository root tidak ditemukan dari current working directory")

ROOT = find_repo_root()
OUTPUT_DIR = ROOT / "output" / "phase5"
REPORT_DIR = ROOT / "reports"
MODEL_DIR = ROOT / "models"
for directory in [OUTPUT_DIR, REPORT_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Repository: {ROOT}")
print(f"Output: {OUTPUT_DIR}")
print(f"Seed: {SEED}")

Repository: /home/andry-alfisi/Project/Project/Buzzer-Detection-
Output: /home/andry-alfisi/Project/Project/Buzzer-Detection-/output/phase5
Seed: 42


/home/andry-alfisi/.config/matplotlib is not a writable directory
Matplotlib created a temporary cache directory at /tmp/matplotlib-t3dudc4p because there was an issue with the default path (/home/andry-alfisi/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


## 2. Load label dan feature Batch 2 + Batch 3

Feature Batch 2 berada di `data/features/` sesuai struktur repository saat ini. Join dilakukan berdasarkan `username` karena label Batch 2 belum memiliki `account_id`; Batch 3 memiliki keduanya. Setiap baris hasil join mewakili satu akun yang telah diberi label.

In [2]:
ID_COLUMNS = [
    "account_id", "post_id", "id", "user_id", "source_post_id", "source_account_id",
    "target_post_id", "target_account_id", "conversation_id"
]

def normalize_identifier_columns(frame):
    frame = frame.copy()
    for column in ID_COLUMNS:
        if column in frame.columns:
            frame[column] = (
                frame[column]
                .astype("string")
                .str.strip()
                .str.replace(r"\.0$", "", regex=True)
            )
    return frame

def read_csv(path):
    if not path.exists():
        raise FileNotFoundError(f"File tidak ditemukan: {path}")
    return normalize_identifier_columns(pd.read_csv(path))

def post_level_aggregates(post_path):
    if not post_path.exists():
        return pd.DataFrame(columns=["username", "url_rate", "avg_words"]), None
    posts = read_csv(post_path)
    if "username" not in posts.columns:
        return pd.DataFrame(columns=["username", "url_rate", "avg_words"]), posts
    posts["username"] = posts["username"].astype("string").str.strip()
    has_url = pd.to_numeric(posts.get("has_url", pd.Series(0, index=posts.index)), errors="coerce").fillna(0)
    body = posts.get("body_clean", posts.get("text", pd.Series("", index=posts.index)))
    posts["_has_url_num"] = has_url
    posts["_word_count"] = body.fillna("").astype(str).str.split().str.len()
    aggregates = (
        posts.dropna(subset=["username"])
        .groupby("username", as_index=False)
        .agg(url_rate=("_has_url_num", "mean"), avg_words=("_word_count", "mean"))
    )
    return aggregates, posts

def derive_component_sizes(posts, account_ids):
    graph = nx.Graph()
    normalized_accounts = account_ids.dropna().astype("string").tolist()
    graph.add_nodes_from(normalized_accounts)
    if posts is not None and {"source_account_id", "target_account_id"}.issubset(posts.columns):
        edges = posts[["source_account_id", "target_account_id"]].dropna()
        for source, target in edges.itertuples(index=False):
            source = str(source).replace(".0", "")
            target = str(target).replace(".0", "")
            if source and target and source != target:
                graph.add_edge(source, target)
    component_by_account = {}
    for component in nx.connected_components(graph):
        size = len(component)
        for account_id in component:
            component_by_account[account_id] = size
    return account_ids.map(component_by_account).fillna(1).astype(float)

BATCH_CONFIGS = {
    "batch2": {
        "label_path": ROOT / "data" / "labels" / "batch2_labeled_combined.csv",
        "feature_path": ROOT / "data" / "features" / "batch2_account_features_with_scores.csv",
        "post_path": ROOT / "data" / "features" / "batch2_posts_with_features.csv",
    },
    "batch3": {
        "label_path": ROOT / "data" / "labels" / "batch3_labeled_combined.csv",
        "feature_path": ROOT / "data" / "features" / "batch3_account_features_with_scores.csv",
        "post_path": ROOT / "data" / "features" / "batch3_posts_with_features.csv",
    },
}

def load_labeled_batch(batch_name, config):
    labels = read_csv(config["label_path"])
    labels["username"] = labels["username"].astype("string").str.strip()
    labels["label"] = pd.to_numeric(labels["label"], errors="coerce")
    labels = labels.dropna(subset=["username", "label"]).copy()
    labels["label"] = labels["label"].astype(int)
    if labels["username"].duplicated().any():
        raise ValueError(f"Duplicate username pada label {batch_name}")

    features = read_csv(config["feature_path"])
    features["username"] = features["username"].astype("string").str.strip()
    if features["username"].duplicated().any():
        raise ValueError(f"Duplicate username pada feature {batch_name}")

    joined = labels.merge(features, on="username", how="left", suffixes=("_label", ""))
    if "account_id_label" in joined.columns:
        joined["account_id"] = joined["account_id"].fillna(joined["account_id_label"])
        joined = joined.drop(columns=["account_id_label"])
    missing_features = joined["account_id"].isna() | joined["n_posts"].isna()
    if missing_features.any():
        missing_usernames = joined.loc[missing_features, "username"].tolist()
        raise ValueError(f"Label {batch_name} gagal join ke feature: {missing_usernames}")

    post_aggregates, posts = post_level_aggregates(config["post_path"])
    joined = joined.merge(post_aggregates, on="username", how="left")
    joined["url_rate"] = pd.to_numeric(joined["url_rate"], errors="coerce").fillna(0.0)
    joined["avg_words"] = pd.to_numeric(joined["avg_words"], errors="coerce").fillna(0.0)

    joined["account_id"] = joined["account_id"].astype("string")
    joined["dataset_id"] = batch_name
    joined["account_group"] = joined["dataset_id"] + "::" + joined["account_id"]
    joined["first_post"] = pd.to_datetime(joined["first_post"], errors="coerce", utc=True)
    joined["last_post"] = pd.to_datetime(joined["last_post"], errors="coerce", utc=True)
    joined["total_degree"] = (
        pd.to_numeric(joined.get("in_degree", 0), errors="coerce").fillna(0)
        + pd.to_numeric(joined.get("out_degree", 0), errors="coerce").fillna(0)
    )
    derived_components = derive_component_sizes(posts, joined["account_id"])
    if "component_size" not in joined.columns:
        joined["component_size"] = derived_components
    else:
        joined["component_size"] = pd.to_numeric(joined["component_size"], errors="coerce").fillna(derived_components)
    return joined

frames = [load_labeled_batch(name, config) for name, config in BATCH_CONFIGS.items()]
labeled = pd.concat(frames, ignore_index=True, sort=False)
labeled["label"] = labeled["label"].astype(int)

print("Loaded labeled accounts:", len(labeled))
print("Columns added from post-level features: url_rate, avg_words")
print(labeled[["dataset_id", "username", "label", "last_post", "url_rate", "avg_words"]].head())

Loaded labeled accounts: 100
Columns added from post-level features: url_rate, avg_words
  dataset_id         username  label                 last_post  url_rate  avg_words
0     batch2      posrondart9      1 2026-11-04 03:11:47+00:00       0.0       28.0
1     batch2        asaprosul      1 2026-10-16 08:05:45+00:00       0.0       13.0
2     batch2      FITRI688212      1 2026-08-29 12:47:43+00:00       1.0       17.0
3     batch2        mzk716524      1 2026-11-06 12:43:29+00:00       1.0       12.0
4     batch2  bukan_fufufafa_      1 2026-09-07 06:29:25+00:00       0.0      250.0


## 3. Audit dataset dan label

Audit ini memastikan tidak ada akun duplikat, semua label biner, dan distribusi label per batch terlihat sebelum modeling. `buzzer_score` hanya ditampilkan sebagai metadata untuk mengingatkan bahwa skor heuristik bukan ground truth.

In [3]:
if labeled["account_group"].duplicated().any():
    raise ValueError("Ada account_group duplikat setelah join")
if not set(labeled["label"].unique()).issubset({0, 1}):
    raise ValueError("Label harus 0 atau 1")

summary = (
    labeled.groupby("dataset_id")
    .agg(accounts=("username", "nunique"), organic=("label", lambda s: int((s == 0).sum())), buzzer=("label", lambda s: int((s == 1).sum())))
    .reset_index()
)
summary["buzzer_rate"] = summary["buzzer"] / summary["accounts"]
print("=== Dataset summary ===")
display(summary.round(3))
print("\nTotal label:", labeled["label"].value_counts().sort_index().to_dict())
print("Unique account groups:", labeled["account_group"].nunique())
print("Missing last_post:", int(labeled["last_post"].isna().sum()))
print("\nCatatan: satu baris merepresentasikan satu akun; group split tetap dipakai untuk mendokumentasikan pencegahan account leakage.")

=== Dataset summary ===
  dataset_id  accounts  organic  buzzer  buzzer_rate
0     batch2        50       27      23         0.46
1     batch3        50       21      29         0.58

Total label: {0: 48, 1: 52}
Unique account groups: 100
Missing last_post: 0

Catatan: satu baris merepresentasikan satu akun; group split tetap dipakai untuk mendokumentasikan pencegahan account leakage.


## 4. Definisi kelompok fitur

- **NLP-only**: agregat isi dan gaya konten yang tersedia dari Phase 3, tanpa `buzzer_score`.
- **Temporal-only**: intensitas, durasi aktif, dan komposisi jenis posting.
- **Graph-only**: posisi dan keterlibatan akun di jaringan interaksi.
- **Combined**: gabungan ketiga kelompok.

`buzzer_score` dan kolom label tidak pernah masuk ke matriks fitur.

In [4]:
FEATURE_GROUPS = {
    "NLP-only": ["avg_tags", "avg_body_len", "url_rate", "avg_words"],
    "Temporal-only": ["n_posts", "active_hours", "post_rate", "n_replies", "n_quotes", "n_originals"],
    "Graph-only": ["in_degree", "out_degree", "total_degree", "pagerank", "component_size"],
}
FEATURE_GROUPS["Combined"] = list(dict.fromkeys(sum(FEATURE_GROUPS.values(), [])))

for group_name, columns in FEATURE_GROUPS.items():
    missing = [column for column in columns if column not in labeled.columns]
    if missing:
        raise ValueError(f"Feature group {group_name} kehilangan kolom: {missing}")
    print(f"{group_name}: {len(columns)} features -> {columns}")

EXCLUDED_FROM_MODEL = ["buzzer_score", "label", "notes", "evidence", "review_status", "candidate_type"]
print("\nExcluded metadata/target:", EXCLUDED_FROM_MODEL)

NLP-only: 4 features -> ['avg_tags', 'avg_body_len', 'url_rate', 'avg_words']
Temporal-only: 6 features -> ['n_posts', 'active_hours', 'post_rate', 'n_replies', 'n_quotes', 'n_originals']
Graph-only: 5 features -> ['in_degree', 'out_degree', 'total_degree', 'pagerank', 'component_size']
Combined: 15 features -> ['avg_tags', 'avg_body_len', 'url_rate', 'avg_words', 'n_posts', 'active_hours', 'post_rate', 'n_replies', 'n_quotes', 'n_originals', 'in_degree', 'out_degree', 'total_degree', 'pagerank', 'component_size']

Excluded metadata/target: ['buzzer_score', 'label', 'notes', 'evidence', 'review_status', 'candidate_type']


## 5. Fungsi model dan metrics

Evaluasi utama menggunakan `StratifiedGroupKFold` pada `account_group`. OOF prediction digabungkan agar confusion matrix dan metrics dihitung dari prediksi akun yang tidak ikut dilatih pada fold tersebut. ROC-AUC ditandai `NaN` bila test set hanya memiliki satu kelas.

In [5]:
def build_estimator(model_type):
    if model_type == "dummy":
        estimator = DummyClassifier(strategy="most_frequent", random_state=SEED)
        return Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", estimator)])
    if model_type == "logistic":
        estimator = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", estimator),
        ])
    if model_type == "rf":
        estimator = RandomForestClassifier(
            n_estimators=300,
            max_depth=6,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1,
        )
        return Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", estimator)])
    raise ValueError(f"Unknown model_type: {model_type}")

def numeric_matrix(frame, columns):
    return frame[columns].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)

def positive_probability(estimator, matrix):
    if hasattr(estimator, "predict_proba"):
        probabilities = estimator.predict_proba(matrix)
        classifier = estimator.named_steps.get("model", estimator) if hasattr(estimator, "named_steps") else estimator
        classes = getattr(classifier, "classes_", np.array([0, 1]))
        if 1 in classes:
            return probabilities[:, list(classes).index(1)]
        return np.ones(len(matrix)) if len(classes) == 1 and classes[0] == 1 else np.zeros(len(matrix))
    if hasattr(estimator, "decision_function"):
        return estimator.decision_function(matrix)
    return estimator.predict(matrix).astype(float)

def calculate_metrics(y_true, y_pred, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    y_score = np.asarray(y_score, dtype=float)
    both_classes = len(np.unique(y_true)) == 2
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_buzzer": precision_score(y_true, y_pred, zero_division=0),
        "recall_buzzer": recall_score(y_true, y_pred, zero_division=0),
        "f1_buzzer": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_score) if both_classes else np.nan,
        "pr_auc": average_precision_score(y_true, y_score) if both_classes else np.nan,
        "support": len(y_true),
        "positive_support": int((y_true == 1).sum()),
        "negative_support": int((y_true == 0).sum()),
    }

def evaluate_holdout(train_frame, test_frame, feature_columns, model_type="rf"):
    estimator = build_estimator(model_type)
    x_train = numeric_matrix(train_frame, feature_columns)
    x_test = numeric_matrix(test_frame, feature_columns)
    y_train = train_frame["label"].astype(int)
    y_test = test_frame["label"].astype(int)
    estimator.fit(x_train, y_train)
    y_pred = estimator.predict(x_test).astype(int)
    y_score = positive_probability(estimator, x_test)
    metrics = calculate_metrics(y_test, y_pred, y_score)
    metrics["confusion_matrix"] = confusion_matrix(y_test, y_pred, labels=[0, 1])
    return metrics, estimator, y_pred, y_score

def evaluate_group_cv(frame, feature_columns, model_type="rf", n_splits=5):
    x_all = numeric_matrix(frame, feature_columns)
    y_all = frame["label"].astype(int).reset_index(drop=True)
    groups = frame["account_group"].reset_index(drop=True)
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof_pred = np.full(len(frame), -1, dtype=int)
    oof_score = np.full(len(frame), np.nan, dtype=float)
    fold_rows = []
    for fold_number, (train_idx, test_idx) in enumerate(splitter.split(x_all, y_all, groups), start=1):
        estimator = build_estimator(model_type)
        estimator.fit(x_all.iloc[train_idx], y_all.iloc[train_idx])
        fold_pred = estimator.predict(x_all.iloc[test_idx]).astype(int)
        fold_score = positive_probability(estimator, x_all.iloc[test_idx])
        oof_pred[test_idx] = fold_pred
        oof_score[test_idx] = fold_score
        fold_metric = calculate_metrics(y_all.iloc[test_idx], fold_pred, fold_score)
        fold_metric["fold"] = fold_number
        fold_rows.append(fold_metric)
    if (oof_pred < 0).any():
        raise RuntimeError("Tidak semua akun mendapatkan OOF prediction")
    metrics = calculate_metrics(y_all, oof_pred, oof_score)
    metrics["confusion_matrix"] = confusion_matrix(y_all, oof_pred, labels=[0, 1])
    fold_frame = pd.DataFrame(fold_rows)
    metrics["fold_f1_mean"] = fold_frame["f1_buzzer"].mean()
    metrics["fold_f1_std"] = fold_frame["f1_buzzer"].std(ddof=0)
    return metrics, oof_pred, oof_score, fold_frame

## 6. Group split: perbandingan baseline

Model yang dibandingkan: majority classifier sebagai baseline paling sederhana, Logistic Regression sebagai baseline linear, dan Random Forest untuk tiap kelompok fitur. Semua model memakai preprocessing imputer di dalam pipeline sehingga statistik imputasi tidak bocor dari test fold.

In [6]:
MODEL_SPECS = [
    {"model": "Majority baseline", "feature_group": "None", "columns": FEATURE_GROUPS["Combined"], "model_type": "dummy"},
    {"model": "NLP-only RF", "feature_group": "NLP-only", "columns": FEATURE_GROUPS["NLP-only"], "model_type": "rf"},
    {"model": "Temporal-only RF", "feature_group": "Temporal-only", "columns": FEATURE_GROUPS["Temporal-only"], "model_type": "rf"},
    {"model": "Graph-only RF", "feature_group": "Graph-only", "columns": FEATURE_GROUPS["Graph-only"], "model_type": "rf"},
    {"model": "Combined Logistic Regression", "feature_group": "Combined", "columns": FEATURE_GROUPS["Combined"], "model_type": "logistic"},
    {"model": "Combined RF", "feature_group": "Combined", "columns": FEATURE_GROUPS["Combined"], "model_type": "rf"},
]

group_cv_rows = []
group_cv_confusions = {}
oof_predictions = {}
for spec in MODEL_SPECS:
    metrics, oof_pred, oof_score, fold_frame = evaluate_group_cv(
        labeled, spec["columns"], model_type=spec["model_type"], n_splits=5
    )
    row = {
        "split": "group_cv",
        "model": spec["model"],
        "feature_group": spec["feature_group"],
        **{key: value for key, value in metrics.items() if key != "confusion_matrix"},
    }
    group_cv_rows.append(row)
    group_cv_confusions[spec["model"]] = metrics["confusion_matrix"]
    oof_predictions[spec["model"]] = pd.DataFrame({
        "account_group": labeled["account_group"],
        "dataset_id": labeled["dataset_id"],
        "username": labeled["username"],
        "label": labeled["label"],
        "predicted_label": oof_pred,
        "predicted_probability": oof_score,
        "model": spec["model"],
    })

group_cv_results = pd.DataFrame(group_cv_rows)
group_cv_results.to_csv(OUTPUT_DIR / "group_cv_metrics.csv", index=False)
print("=== GROUP CV RESULTS ===")
display(group_cv_results.round(3))

=== GROUP CV RESULTS ===
      split                         model  feature_group  accuracy  precision_buzzer  recall_buzzer  f1_buzzer  roc_auc  pr_auc  support  \
0  group_cv             Majority baseline           None      0.52             0.520          1.000      0.684    0.500   0.520      100   
1  group_cv                   NLP-only RF       NLP-only      0.73             0.727          0.769      0.748    0.779   0.771      100   
2  group_cv              Temporal-only RF  Temporal-only      0.53             0.542          0.615      0.577    0.550   0.532      100   
3  group_cv                 Graph-only RF     Graph-only      0.73             0.712          0.808      0.757    0.660   0.614      100   
4  group_cv  Combined Logistic Regression       Combined      0.74             0.741          0.769      0.755    0.772   0.760      100   
5  group_cv                   Combined RF       Combined      0.73             0.736          0.750      0.743    0.793   0.796      10

In [7]:
metric_columns = ["accuracy", "precision_buzzer", "recall_buzzer", "f1_buzzer", "roc_auc", "pr_auc"]
plot_data = group_cv_results.set_index("model")[metric_columns]
ax = plot_data.plot(kind="bar", figsize=(14, 6), ylim=(0, 1.05), color=["#4c78a8", "#f58518", "#54a24b", "#e45756", "#72b7b2", "#b279a2"] )
ax.set_title("Phase 5 - Group CV: Comparison Baseline")
ax.set_ylabel("Score")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=30)
ax.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "group_cv_metrics_comparison.png", dpi=180, bbox_inches="tight")
plt.close()

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for axis, spec in zip(axes.flat, MODEL_SPECS):
    cm = group_cv_confusions[spec["model"]]
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axis, xticklabels=["Organic", "Buzzer"], yticklabels=["Organic", "Buzzer"])
    axis.set_title(spec["model"])
    axis.set_xlabel("Predicted")
    axis.set_ylabel("Actual")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "group_cv_confusion_matrices.png", dpi=180, bbox_inches="tight")
plt.close()
print("Saved group CV visualizations")

Saved group CV visualizations


## 7. Temporal split

Akun diurutkan berdasarkan `last_post`. 70% waktu paling awal dipakai untuk train dan 30% waktu berikutnya untuk test. Split ini menguji apakah pola dari waktu sebelumnya dapat digeneralisasi ke periode berikutnya; hasilnya tidak boleh dibaca sebagai cross-validation.

In [8]:
temporal_frame = labeled.dropna(subset=["last_post"]).sort_values("last_post").reset_index(drop=True)
split_index = int(np.floor(len(temporal_frame) * 0.70))
split_index = max(1, min(split_index, len(temporal_frame) - 1))
temporal_train = temporal_frame.iloc[:split_index].copy()
temporal_test = temporal_frame.iloc[split_index:].copy()
cutoff_before = temporal_train["last_post"].max()
cutoff_after = temporal_test["last_post"].min()

print(f"Temporal train: {len(temporal_train)} akun, {temporal_train['last_post'].min()} s.d. {cutoff_before}")
print(f"Temporal test:  {len(temporal_test)} akun, {cutoff_after} s.d. {temporal_test['last_post'].max()}")
print("Train labels:", temporal_train["label"].value_counts().sort_index().to_dict())
print("Test labels:", temporal_test["label"].value_counts().sort_index().to_dict())

temporal_specs = [spec for spec in MODEL_SPECS if spec["model"] != "Majority baseline"]
temporal_rows = []
temporal_confusions = {}
for spec in temporal_specs:
    metrics, _, y_pred, y_score = evaluate_holdout(
        temporal_train, temporal_test, spec["columns"], model_type=spec["model_type"]
    )
    row = {
        "split": "temporal_70_30",
        "model": spec["model"],
        "feature_group": spec["feature_group"],
        **{key: value for key, value in metrics.items() if key != "confusion_matrix"},
    }
    temporal_rows.append(row)
    temporal_confusions[spec["model"]] = metrics["confusion_matrix"]

temporal_results = pd.DataFrame(temporal_rows)
temporal_results.to_csv(OUTPUT_DIR / "temporal_metrics.csv", index=False)
print("=== TEMPORAL RESULTS ===")
display(temporal_results.round(3))

Temporal train: 70 akun, 2026-08-29 12:47:43+00:00 s.d. 2026-11-07 03:09:10+00:00
Temporal test:  30 akun, 2026-11-07 03:13:24+00:00 s.d. 2026-11-08 11:41:03+00:00
Train labels: {0: 36, 1: 34}
Test labels: {0: 12, 1: 18}
=== TEMPORAL RESULTS ===
            split                         model  feature_group  accuracy  precision_buzzer  recall_buzzer  f1_buzzer  roc_auc  pr_auc  \
0  temporal_70_30                   NLP-only RF       NLP-only     0.500             0.636          0.389      0.483    0.616   0.744   
1  temporal_70_30              Temporal-only RF  Temporal-only     0.400             0.500          0.278      0.357    0.512   0.668   
2  temporal_70_30                 Graph-only RF     Graph-only     0.667             0.750          0.667      0.706    0.738   0.794   
3  temporal_70_30  Combined Logistic Regression       Combined     0.567             0.778          0.389      0.519    0.833   0.795   
4  temporal_70_30                   Combined RF       Combined     0.

In [9]:
ax = temporal_results.set_index("model")[metric_columns].plot(kind="bar", figsize=(13, 6), ylim=(0, 1.05), color=["#4c78a8", "#f58518", "#54a24b", "#e45756", "#72b7b2", "#b279a2"])
ax.set_title("Phase 5 - Temporal Holdout (70% earlier -> 30% later)")
ax.set_ylabel("Score")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=25)
ax.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "temporal_metrics_comparison.png", dpi=180, bbox_inches="tight")
plt.close()

combined_temporal_cm = temporal_confusions["Combined RF"]
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(combined_temporal_cm, annot=True, fmt="d", cmap="Oranges", cbar=False, ax=ax, xticklabels=["Organic", "Buzzer"], yticklabels=["Organic", "Buzzer"])
ax.set_title("Temporal Holdout - Combined RF")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "temporal_combined_rf_confusion_matrix.png", dpi=180, bbox_inches="tight")
plt.close()
print("Saved temporal visualizations")

Saved temporal visualizations


## 8. Cross-issue evaluation

Karena Batch 2 dan Batch 3 berasal dari isu berbeda, evaluasi ini dipisahkan dari hasil gabungan. Model dilatih pada satu issue lalu diuji pada issue lain untuk melihat domain shift.

In [10]:
cross_issue_rows = []
cross_issue_confusions = {}
for train_batch, test_batch in [("batch2", "batch3"), ("batch3", "batch2")]:
    train_frame = labeled[labeled["dataset_id"] == train_batch].copy()
    test_frame = labeled[labeled["dataset_id"] == test_batch].copy()
    metrics, _, _, _ = evaluate_holdout(
        train_frame, test_frame, FEATURE_GROUPS["Combined"], model_type="rf"
    )
    row = {
        "split": "cross_issue",
        "train_batch": train_batch,
        "test_batch": test_batch,
        **{key: value for key, value in metrics.items() if key != "confusion_matrix"},
    }
    cross_issue_rows.append(row)
    cross_issue_confusions[f"{train_batch}_to_{test_batch}"] = metrics["confusion_matrix"]

cross_issue_results = pd.DataFrame(cross_issue_rows)
cross_issue_results.to_csv(OUTPUT_DIR / "cross_issue_metrics.csv", index=False)
print("=== CROSS-ISSUE RESULTS (Combined RF) ===")
display(cross_issue_results.round(3))

=== CROSS-ISSUE RESULTS (Combined RF) ===
         split train_batch test_batch  accuracy  precision_buzzer  recall_buzzer  f1_buzzer  roc_auc  pr_auc  support  positive_support  \
0  cross_issue      batch2     batch3      0.42               0.5          0.138      0.216    0.531   0.609       50                29   
1  cross_issue      batch3     batch2      0.42               0.2          0.087      0.121    0.609   0.491       50                23   

   negative_support  
0                21  
1                27  


In [11]:
cross_plot = cross_issue_results.set_index("test_batch")[metric_columns].plot(kind="bar", figsize=(12, 6), ylim=(0, 1.05), color=["#4c78a8", "#f58518", "#54a24b", "#e45756", "#72b7b2", "#b279a2"])
cross_plot.set_title("Phase 5 - Cross-Issue Generalization (Combined RF)")
cross_plot.set_ylabel("Score")
cross_plot.set_xlabel("Test batch; train batch ditampilkan di tabel")
cross_plot.tick_params(axis="x", rotation=0)
cross_plot.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "cross_issue_metrics_comparison.png", dpi=180, bbox_inches="tight")
plt.close()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for axis, (name, cm) in zip(axes, cross_issue_confusions.items()):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", cbar=False, ax=axis, xticklabels=["Organic", "Buzzer"], yticklabels=["Organic", "Buzzer"])
    axis.set_title(name.replace("_", " -> "))
    axis.set_xlabel("Predicted")
    axis.set_ylabel("Actual")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "cross_issue_confusion_matrices.png", dpi=180, bbox_inches="tight")
plt.close()
print("Saved cross-issue visualizations")

Saved cross-issue visualizations


## 9. Fit artifact final dan simpan OOF prediction

Artifact ini dilatih pada seluruh 100 akun berlabel hanya untuk kebutuhan baseline/inference setelah evaluasi. Angka performa yang dilaporkan tetap berasal dari group CV, temporal holdout, dan cross-issue; bukan dari skor training penuh.

In [12]:
final_feature_columns = FEATURE_GROUPS["Combined"]
final_model = build_estimator("rf")
final_model.fit(numeric_matrix(labeled, final_feature_columns), labeled["label"].astype(int))

artifact = {
    "model": final_model,
    "feature_columns": final_feature_columns,
    "feature_groups": FEATURE_GROUPS,
    "label_mapping": {0: "Organic", 1: "Buzzer"},
    "training_rows": len(labeled),
    "training_batches": sorted(labeled["dataset_id"].unique().tolist()),
    "random_seed": SEED,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "note": "Phase 5 baseline; buzzer_score excluded from model features and is not ground truth.",
}
model_path = MODEL_DIR / "phase5_baseline_rf.pkl"
with model_path.open("wb") as handle:
    pickle.dump(artifact, handle)

oof_export = pd.concat(oof_predictions.values(), ignore_index=True)
oof_export.to_csv(OUTPUT_DIR / "group_cv_oof_predictions.csv", index=False)
print(f"Saved model: {model_path}")
print(f"Saved OOF predictions: {OUTPUT_DIR / 'group_cv_oof_predictions.csv'}")

Saved model: /home/andry-alfisi/Project/Project/Buzzer-Detection-/models/phase5_baseline_rf.pkl
Saved OOF predictions: /home/andry-alfisi/Project/Project/Buzzer-Detection-/output/phase5/group_cv_oof_predictions.csv


## 10. Generate baseline evaluation report

Report menyimpan metodologi, perbandingan metrics, keterbatasan selection bias, keterbatasan label, dan rekomendasi menuju Phase 6.

In [13]:
def markdown_table(frame, columns=None):
    selected = frame[columns] if columns else frame
    return selected.to_markdown(index=False, floatfmt=".3f")

report_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")
report_text = f"""# COORDINX - PHASE 5 BASELINE EVALUATION REPORT

**Generated (UTC):** {report_date}
**Notebook:** `notebooks/04b_supervised_baseline.ipynb`

## 1. Dataset

{markdown_table(summary, ['dataset_id', 'accounts', 'organic', 'buzzer', 'buzzer_rate'])}

Total akun berlabel: **{len(labeled)}**. Setiap baris mewakili satu akun; `account_group` digunakan untuk group split.

## 2. Feature groups

- **NLP-only:** `{', '.join(FEATURE_GROUPS['NLP-only'])}`
- **Temporal-only:** `{', '.join(FEATURE_GROUPS['Temporal-only'])}`
- **Graph-only:** `{', '.join(FEATURE_GROUPS['Graph-only'])}`
- **Combined:** seluruh feature dari tiga kelompok di atas
- `buzzer_score` tidak dimasukkan sebagai fitur dan tidak dianggap sebagai ground truth.

## 3. Group CV result

{markdown_table(group_cv_results, ['model', 'feature_group', 'accuracy', 'precision_buzzer', 'recall_buzzer', 'f1_buzzer', 'roc_auc', 'pr_auc', 'fold_f1_mean', 'fold_f1_std'])}

Group CV menggunakan `StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)`. Karena dataset ini satu baris per akun, group split secara matematis mirip account-level stratified CV, tetapi tetap dipakai untuk mencegah pola multi-baris per akun bila dataset bertambah.

## 4. Temporal holdout result

Split temporal: train pada {len(temporal_train)} akun paling awal dan test pada {len(temporal_test)} akun berikutnya. Batas waktu train: `{cutoff_before}`; awal test: `{cutoff_after}`.

{markdown_table(temporal_results, ['model', 'feature_group', 'accuracy', 'precision_buzzer', 'recall_buzzer', 'f1_buzzer', 'roc_auc', 'pr_auc'])}

## 5. Cross-issue result

{markdown_table(cross_issue_results, ['train_batch', 'test_batch', 'accuracy', 'precision_buzzer', 'recall_buzzer', 'f1_buzzer', 'roc_auc', 'pr_auc'])}

Hasil cross-issue dibaca sebagai uji generalisasi domain, bukan sebagai pengganti evaluasi in-domain.

## 6. Artifacts

- `output/phase5/group_cv_metrics.csv`
- `output/phase5/group_cv_metrics_comparison.png`
- `output/phase5/group_cv_confusion_matrices.png`
- `output/phase5/temporal_metrics.csv`
- `output/phase5/temporal_metrics_comparison.png`
- `output/phase5/temporal_combined_rf_confusion_matrix.png`
- `output/phase5/cross_issue_metrics.csv`
- `output/phase5/cross_issue_metrics_comparison.png`
- `output/phase5/cross_issue_confusion_matrices.png`
- `models/phase5_baseline_rf.pkl`

## 7. Limitations

1. **Selection bias:** label Phase 4 berasal dari Top 30 suspicious + Random 20, bukan sample populasi yang sepenuhnya acak.
2. **Label size:** hanya 100 akun dari dua issue; hasil metrics belum cukup untuk klaim performa produksi.
3. **Annotation quality:** label saat ini single-reviewer/human-in-the-loop dan belum memiliki multi-annotator agreement seperti Cohen's Kappa.
4. **Sparse temporal signal:** banyak akun hanya memiliki satu posting sehingga `active_hours` dan `post_rate` memiliki keterbatasan interpretasi.
5. **Domain shift:** distribusi teks, waktu, dan jaringan Batch 2 vs Batch 3 berbeda.
6. **NLP scope:** NLP-only Phase 5 masih memakai agregat handcrafted, belum IndoBERTweet.
7. **No causal claim:** feature importance atau metrics tidak membuktikan bahwa suatu akun benar-benar buzzer di luar definisi label anotasi.

## 8. Recommendation for Phase 6

Gunakan split dan report Phase 5 sebagai baseline tetap. Phase 6 dapat menambahkan embedding NLP/IndoBERTweet, evaluasi label quality yang lebih kuat, tuning yang nested atau time-aware, dan analisis error pada false positive/false negative tanpa memasukkan `buzzer_score` sebagai label.
"""
report_path = REPORT_DIR / "BASELINE_EVALUATION_REPORT.md"
report_path.write_text(report_text, encoding="utf-8")
print(f"Saved report: {report_path}")
print("\nPhase 5 selesai: group CV, temporal holdout, cross-issue, visualisasi, report, dan model artifact tersedia.")

Saved report: /home/andry-alfisi/Project/Project/Buzzer-Detection-/reports/BASELINE_EVALUATION_REPORT.md

Phase 5 selesai: group CV, temporal holdout, cross-issue, visualisasi, report, dan model artifact tersedia.
